# Lab 4 — Clean the broken sensor feed
### EP0418 · Topic 4 · 2 practical hours

**You need:** nothing to download: the notebook fetches `ep0418.py` and
`smart_building_sensors.csv` itself. Have the printed Lab 4 sheet beside you.

## How the cells work

- **Grey code cells:** run each one when you reach it, top to bottom. Click in it
  and press Shift+Enter.
- **Text boxes** have a bold label ending in a colon, like **My answers:**.
  Double-click, type, press Shift+Enter. Or write on the printed sheet.
- **Gemini's code** arrives as a **green cell**. Click **Accept and run** (bottom of
  the chat panel), then run the task's check cell.
- **Check cells** start with `# TODO(you):`, which says what to run first. Never
  change them: if Gemini shows red and green lines inside one, click **Cancel**
  (red and green lines inside Gemini's own green cell are fine). Leave lines like
  `s = keep("s")` as they are. If a check cell says *Not there yet* or *No ... yet*,
  type in the chat *call the result* and the name the task asks for, accept, and
  run the check cell again.
- **Gemini's lab rules** (*EP0418 lab rules*) are built in. If a box *Review custom
  instructions* appears, click **Use custom instruction**. The sliders icon beside
  **+** in the chat box should be blue; if it is grey, click it and choose *EP0418
  lab rules*. Keep *Learn mode* off.
- **If Gemini says *Daily limit reached*:** paste the same prompt into
  ChatGPT, Claude or Copilot in a new tab. It cannot see your notebook: copy its code
  into a new cell (**+ Code** at the top of the page) and run it, and paste any
  failed check or red error back into its chat. Do the quiz last to save your limit.
- **Prompts** in grey quote boxes start with **Rules:** (for example, do not drop
  rows; show the row count before and after), then **Task:**. Keep the Rules when
  you change a prompt. A prompt with no example is **zero-shot**; one example of the
  output you want makes it **one-shot**, several make it **few-shot**.
- **Checks** (the `check.*` lines under a task) are the **verification contract**.
  Do not edit them. In industry they are called **evals**.
- **A failing check** says what it found and what to try. Failing checks are normal
  while you work.


> ### This dataset is deliberately broken
> **Six** separate faults are hidden in this file: −999 values that are not
> readings, duplicate timestamps, a five-and-a-half-hour outage, an impossible energy spike, a
> sensor that gets stuck, and a sensor that slowly drifts. The table and the checks
> show you the first four. **The last two you will only find by plotting**, in
> Task 5.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. If a box *Review custom instructions* appears while you save the copy, click **Make a copy**: it keeps the lab's short rules for the AI assistant. Colab opens your copy in a new tab with *Copy of* at the start of its name at the top left: work in that tab and close the old one. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    global _NAMED
    v = globals().get(name, ...)
    if v is ... or v is None:
        print(f"Not there yet: nothing called {name}. Not run Gemini's green cell yet? Click Accept and "
              f"run on it, then run this cell again. Already ran it? Type in the same chat: call the "
              f"result {name}. Then Accept and run, and run this cell again.")
        _NAMED = True
        return ...
    return v


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    global _NAMED
    if any(v is ... or v is None for v in values):
        if not globals().get("_NAMED"):      # keep() has already said exactly what is missing
            print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
                  "or paste its code into a new cell and run it), then run this cell again.")
        _NAMED = False
        return False
    _NAMED = False
    return True


def show_predict(data, unit):
    """Prediction League: run the newest predict that Gemini wrote and print its code to copy."""
    ran = globals().get("In", [])                                   # every cell you have run, oldest first
    mine = [c for c in ran if "def predict(" in c and "The baseline predicts" not in c and "show_predict" not in c]
    if not mine:
        print("Gemini's predict has not run yet. Find the cell Gemini added in Step 2 (it starts with")
        print("def predict, or with import lines just above it), click in it, press Shift+Enter,")
        print("then run this cell again."); return
    code = mine[-1].strip()
    for short, line in (("np.", "import numpy as np"), ("pd.", "import pandas as pd")):
        if short in code and line not in code:                     # the league page starts empty
            code = line + "\n" + code
    ns = {}
    exec(code, ns)
    print("Your predict says", round(ns["predict"](data.copy()), 1), unit)
    print()
    print("----- copy from the line below -----")
    print(code)
    print("----- to the line above -----")

CSV = "smart_building_sensors.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])
print(f"{len(raw):,} rows, {raw.timestamp.min()} -> {raw.timestamp.max()}")

## First look — what is in the file (10 minutes)

The file comes from one classroom's sensor logger. The cell above printed its size:
17,354 rows, one every 5 minutes from 1 May to 29 June 2026.

**What each field means.** Read this before the first look: you can only spot an
impossible value once you know what each column measures, and in what unit.

| Field | What it means | Example (1 May, 12:15) |
|---|---|---|
| `timestamp` | when the reading was taken, Singapore time, every 5 minutes | 2026-05-01 12:15 |
| `temp_c` | room temperature, in °C | 25.76 |
| `humidity_pct` | how damp the air is, in %: it can only be 0 to 100 | 72.1 |
| `co2_ppm` | carbon dioxide in the air, in parts per million. Outdoor air is about 420; people breathing raise it | 865 |
| `occupancy` | how many people are in the room | 12 |
| `energy_kwh` | electricity the room used in that 5-minute slot, in kWh | 7.668 |

1. **Run the cell below.** It counts blank values and repeated timestamps, then
   prints each column's smallest (`min`), average (`mean`) and largest (`max`) value.

In [ ]:
print("blank values:", raw.isna().sum().sum())
print("duplicated timestamps:", raw.timestamp.duplicated().sum())
print()
print(raw.describe().T[["min", "mean", "max"]].round(1).to_string())

### Read that table before going further · `VERIFY`

**What you should see:** 0 blank values, 140 duplicated timestamps, and a table with
the smallest, mean and largest value of each column. `isna()` reports 0 blanks, but
the smallest `humidity_pct` is −999.

**−999 is not a reading.** Humidity can only be 0 to 100 %. −999 is the logger's
code for "no answer", called a **sentinel** (Topic 4). pandas counts it as a real
number: the 95 values of −999 pull the mean humidity down from 75.8 % to the 69.9 %
in the table. And `isna()` will never find them, because those cells are not empty.

**How one −999 drags an average**, on six real readings from 1 May (it plays slowly,
about a minute):
![Six humidity readings, one of them -999: the mean of all six is impossible](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_sentinel.gif)

1. `VERIFY` **In the table the cell above printed**, compare each column's `max`
   with its `mean`.
2. `VERIFY` **Find** one other column whose `min` or `max` is physically
   impossible: a value this room could not produce. Hint: a classroom does not use
   eight times its usual energy in five minutes.
3. **Write it** in the box below: the column, min or max, and why it is impossible.
   You do not fix it today: Lab 5 is about spikes.

**Other impossible values (column, min or max, why impossible):**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

One good answer: `energy_kwh`, the maximum. It reaches 33.6 kWh in a five-minute
slot, while the mean is 4.3 and the normal daily peak in this room is about 9.2 (the
middle of the 60 daily maximums). That is about eight times the mean and more than
three and a half times the normal peak, and a classroom cannot do that. If you
plot it later you will see it is not one bad row but three short bursts, on 22 May,
3 June and 18 June. A common wrong answer is the `co2_ppm` maximum of about 1,217
ppm. It looks big next to the mean of 609, but a full, poorly ventilated classroom
really can reach that, so it is high, not impossible. A minimum of 0 for `occupancy`
is also fine: the room is empty at night.
</details>

## Task 1 — Choose what each −999 becomes (15 minutes)

*Topic 4: zero is not the same as missing. A missing reading must look missing.*

**Why change the −999s?** Anything that reads the `humidity_pct` column uses each
−999 as a real humidity. The 95 of them already pull the mean from 75.8 % to 69.9 %,
and a chart or an alarm built on humidity would be wrong in the same way. So each
−999 must become something else before the column is used. What it becomes is the
decision in this task.

**The goal:** `clean`, a copy of `raw` with no −999 left in `humidity_pct`. The tags
show where you are in the loop: `PROMPT` → `GENERATE` → `VERIFY` → `REFINE`.

**Step 1. Predict, before you ask the AI.** On 1 May at 12:20 the humidity is −999,
but the same row also holds a real temperature (26.13 °C), CO2 (852 ppm), occupancy
(11) and energy (7.874 kWh). A −999 can be deleted with its whole row, turned into 0,
or turned into a blank (NaN). Write two short answers in the box below:

1. Which of the three should each −999 become, and why?
2. What goes wrong with each of the other two?

**My plan:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 2.** `PROMPT` **Ask Gemini for the code.** Click the **spark icon** (bottom
centre) to open Gemini and start a new chat (**⋮** › **New chat** › **Yes**). The
prompt below ends with one sentence hidden: what each −999 must become. **Before you
open it, say your answer** (it is question 1 of your plan in Step 1).

<details><summary><b>Show the prompt</b> (click, then copy all of it into the chat and press Enter)</summary>

> **Rules:** Do not remove any rows. Show the row count before and after. Explain in
> plain words what the code does.
>
> **Task:** I have a DataFrame `raw` where -999 in `humidity_pct` means "no reading".
> Make a copy called `clean`. Each -999 in `clean` must become NaN (blank), and its
> row must stay.

**The last sentence was the hidden one.** It says two things. Did your answer say both?

- **NaN, not 0**: humidity 0 % is a real reading (bone-dry air), so 0 would turn "no
  answer" into a false measurement, and `isna()` would still find no blanks;
- **its row stays**: the other four readings in that row are real. Deleting the 95
  rows would throw away 380 good readings (95 rows × 4 columns).
</details>

This prompt is **zero-shot**: it shows no example of the output, so its last sentence
is the only thing that tells Gemini what a −999 must become.

**Step 3.** `GENERATE` **Run Gemini's code.**

1. Gemini answers by adding a new code cell to your notebook, with a green edge.
   Click **Accept and run**: the button sits at the bottom of the chat panel, just
   above the box where you type. This keeps the code and runs it.
2. If the code appears only in the chat and no green cell appears: copy the code,
   click **+ Code** at the top of the page, paste it into the new cell and run it.
3. Colab may jump to the green cell, even at the top of the notebook. Scroll back
   down to Task 1 for Step 4.

**Step 4.** `VERIFY` **Run the check cell just below.** You should see two `[PASS]`
lines (the expected output is under the cell). `rows_at_least` requires
every one of `raw`'s 17,354 rows: you said what "right" looks like before running
anything. That is called a **known-answer eval**.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `clean`), then run this cell.
clean = keep("clean")

# ---- verification contract — do not edit ----
check.no_sentinels(clean, cols=["humidity_pct"], sentinel=-999)
check.rows_at_least(clean, len(raw))
check.report()

**Expected output** *(computed from the data file when this notebook was built)*:

- **First check:** `[PASS] no_sentinels(-999)`: no −999 left in `humidity_pct`.
- **Second check:** `[PASS] rows_at_least(17354)`, and under it *17,354 rows.*
- **Last line:** *All checks passed.*

**Step 5.** `VERIFY` **Run the cell below.** It draws `humidity_pct` twice: the top
panel is `raw`, the bottom panel is your `clean`. Then find your result in the table
under the chart: outcome A, B or C.

In [ ]:
if filled(clean):
    show.before_after(raw, clean, "humidity_pct", title="Replacing -999 with NaN")

**What you should see:** the top panel has thin spikes down to −999. Compare the
bottom panel and its title with the three outcomes:

| | rows in `clean` | bottom panel | what it means |
|---|---|---|---|
| **A** | 17,354 | small gaps where the spikes were | **Right.** No reading now looks like no reading. |
| **B** | 17,259 | no gaps; the title says *after: 17,259 rows* | **Wrong:** the 95 rows were deleted. `rows_at_least` fails. |
| **C** | 17,354 | spikes down to 0 | **Wrong, and no check catches it:** "no answer" became 0 % humidity. |

**If you got A:** go on to Task 2.

**If you got B or C:** `REFINE` ask Gemini to fix it. In the same chat, type one
sentence with your numbers, for example:
- for **B**: *clean has 17,259 rows but must keep all 17,354. Replace -999 with NaN
  instead of removing the rows. Keep everything else.*
- for **C**: *clean has 17,354 rows, but the -999 values became 0. Humidity 0 is a
  real reading; they must be NaN. Fix that and keep everything else.*

Click **Accept and run**, then run the check cell and the chart cell again: the
bottom panel should now have gaps. Give the numbers, not just *it didn't work*:
with the numbers, Gemini knows what to fix; without them, it guesses.

## Task 2 — Remove repeated timestamps and sort by time (10 minutes)

**Why remove them?** A moment that appears twice is counted twice: every count,
average and alarm that uses the rows gives it double weight. Some timestamps appear
twice because the logger sent the same moment again (a retry). On a logger that
retries, the last copy is the corrected one. For example, on 24 May at 03:55 the
first copy had humidity −999 and the second copy 79.4. So you keep the last copy and
drop the others.

**The goal:** `clean` with each timestamp once, sorted by time.

**Step 1.** `PROMPT` **Ask Gemini.** Open Gemini (the **spark icon** at the bottom centre of the Colab window), start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), paste this and press Enter. To copy it, drag
across the quoted text below from its first word to its last, then press Ctrl+C
(Cmd+C on a Mac):

> **Rules:** Do not drop any rows except the duplicate copies. Show the row count
> before and after. Explain in plain words what the code does.
>
> **Task:** `clean` is a DataFrame with a `timestamp` column, and some timestamps
> appear more than once. Remove the duplicates, keeping the last copy of each
> timestamp, sort by timestamp, and keep the result in `clean`.

**Step 2.** `GENERATE` **Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it. If Gemini ever shows red and green lines inside the check cell
(first line `# TODO(you)`), click **Cancel**: check cells are never changed.
Colab scrolls to the green cell, even to the very top of the notebook, so after
**Accept and run** scroll back down to this task for Step 3.

**Step 3.** `VERIFY` **Run the check cell below.** You should see two `[PASS]` lines, and 17,214 rows (140 fewer).

**Step 4.** `REFINE` **Only if a check fails, a number looks wrong, or Gemini's green cell
shows red error text:** in the same chat, type one
sentence saying what you saw and what you expected, for example *sorted_in_time failed; sort clean by timestamp after removing the duplicates.* Press Enter,
click **Accept and run** (Gemini may add a new green cell or change its own earlier
one; either is fine), and run the check cell below again. If everything
passed, there is nothing to fix: move on.

If it is `no_duplicate_timestamps` that fails, type this instead, with the number it
reports: *no_duplicate_timestamps failed: 140 duplicated timestamps are left. Remove every
repeated timestamp, keeping the last copy, and keep everything else.*

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it updates `clean`), then run this cell.
clean = keep("clean")

# ---- verification contract — do not edit ----
check.no_duplicate_timestamps(clean, col="timestamp")
check.sorted_in_time(clean, col="timestamp")
check.report()
if filled(clean):
    print(f"rows in clean: {len(clean):,}")

**What your code in this task just did**, on five real rows from 24 May (it plays
slowly, about a minute):
![Two rows for the same moment: the first copy says -999, the retry 79.4, so keep the last copy](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_duplicates.gif)

**Expected output** *(computed from the data file when this notebook was built)*:

- **First check:** `[PASS] no_duplicate_timestamps`, and under it *All 17,214 timestamps unique.*
- **Second check:** `[PASS] sorted_in_time`.
- **Last line:** *rows in clean: 17,214*, which is 140 fewer than `raw`'s 17,354.

If the last line says 17,074, both copies of each repeated timestamp were dropped. Type in the same chat: *clean has 17,074 rows, expected 17,214. Keep the last copy of each repeated timestamp; do not drop both.*

In the Decision box at the end, write: kept the last copy · the logger retries, so the last copy is the corrected one · rows 17,354 → 17,214.

## Task 3 — Put the readings on a regular 5-minute grid (15 minutes)

**Why a regular grid?** Many later steps count rows to measure time: 12 rows make
an hour only if there is exactly one row every 5 minutes. Right now the hours when
the logger sent nothing have no rows at all, so a count of rows steps over them
without saying so.

**Resample** means: put the readings onto a fixed timetable, one row per 5-minute
slot, even for slots where the sensor sent nothing (those stay blank). You want
`grid`.

**Step 1.** `PROMPT` **Ask Gemini.** Open Gemini (the **spark icon** at the bottom centre of the Colab window), start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), paste this and press Enter. To copy it, drag
across the quoted text below from its first word to its last, then press Ctrl+C
(Cmd+C on a Mac):

> **Rules:** Do not drop empty slots: leave them as NaN. Show the row count before and
> after. Explain in plain words what the code does.
>
> **Task:** `clean` is a DataFrame with a `timestamp` column and the columns temp_c,
> humidity_pct, co2_ppm, energy_kwh and occupancy. Set timestamp as the index and
> resample onto a regular 5-minute grid, taking the mean in each slot. Call the result
> `grid`.

**Step 2.** `GENERATE` **Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it. If Gemini ever shows red and green lines inside the check cell
(first line `# TODO(you)`), click **Cancel**: check cells are never changed.
Colab scrolls to the green cell, even to the very top of the notebook, so after
**Accept and run** scroll back down to this task for Step 3.

**Step 3.** `VERIFY` **Run the check cell below.** You should see two `[PASS]` lines, and 17,280 rows: 66 *more* than `clean`.

**Step 4.** `REFINE` **Only if a check fails, a number looks wrong, or Gemini's green cell
shows red error text:** in the same chat, type one
sentence saying what you saw and what you expected, for example *regular_cadence failed: grid is not every 5 minutes; resample with '5min' on the timestamp index.* Press Enter,
click **Accept and run** (Gemini may add a new green cell or change its own earlier
one; either is fine), and run the check cell below again. If everything
passed, there is nothing to fix: move on.

If it is `values_in_range` that fails, saying the actual range starts at 0, Gemini
filled the empty slots with 0. Type in the same chat: *values_in_range failed: temp_c has 66 values
of 0. Leave the empty slots as NaN; do not fill them. Keep everything else.*

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `grid`), then run this cell.
grid = keep("grid")

# ---- verification contract — do not edit ----
check.regular_cadence(grid, "5min")
check.values_in_range(grid.reset_index() if hasattr(grid, "reset_index") else grid, "temp_c", 15, 40)
check.report()
if filled(grid):
    print(f"rows in grid: {len(grid):,}")

**What resampling just did** to the 11 June outage, slot by slot (it plays slowly,
about a minute):
![The 11 June outage on the 5-minute grid: 66 empty slots become rows of NaN, so 17,214 rows become 17,280](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_grid.gif)

**Expected output** *(computed from the data file when this notebook was built)*:

- **First check:** `[PASS] regular_cadence(5min)`, and under it *All 17,279 gaps are exactly 5min.*
- **Second check:** `[PASS] values_in_range(temp_c)`.
- **Last line:** *rows in grid: 17,280*. The count goes *up* by 66 from `clean`'s 17,214: the empty slots of the 11 June outage now exist as rows of NaN.

**Run the cell below.** It draws `co2_ppm` from `grid`, with the hours of the
11 June outage shaded.

In [ ]:
if filled(grid):
    show.series(grid, col="co2_ppm", title="CO2 on the regular 5-minute grid; shaded: the 11 June outage",
                span=("2026-06-11 02:00", "2026-06-11 07:30"))

**What you should see:** a break in the line inside the shaded period. Before
resampling, those hours simply had no rows and nothing showed. Now they are empty
slots on the grid, and the chart is honest about them. Count them in Task 4: the
row count goes *up* from `clean` to `grid`, and you have to say why.

## Task 4 — Explain every change in the row count (10 minutes)

Task 1 kept every row. The count then changed twice: in Task 2 (`raw` 17,354 →
`clean` 17,214) and in Task 3 (`clean` 17,214 → `grid` 17,280). Accounting for
every row like this is called **error analysis**: find where each difference comes
from, and count it.

1. **In the cell below**, replace the words between the quotes after `EXPLANATION =`
   with your own explanation of both changes: **which** rows and **why**. For example
   (made-up numbers): *"12 rows removed in Task 2 because ...; 30 rows
   added in Task 3 because ..."*. *"Cleaning removed some rows"* will be rejected.
   The two real numbers are in the expected output of Task 2 (the line *rows in
   clean*) and of Task 3 (the line *rows in grid*).
2. `VERIFY` **Run the cell.** You should see `[PASS] rows_accounted_for`. If it says your
   explanation is missing or too short, add the reasons and run it again.

In [ ]:
# TODO(you): replace the words between the quotes with your explanation (see the shape above)
EXPLANATION = "TODO(you): replace with a real explanation"

# ---- verification contract — do not edit ----
check.rows_accounted_for(
    raw, clean, grid,
    names=["raw", "clean", "grid"],
    explain=EXPLANATION,
)
check.report()

## Task 5 — Find the two faults no check can catch (20 minutes)

Two faults in this file pass every check, because every value is in range, none is
missing and none is repeated:

- a sensor that **stops changing**: it sends exactly the same value, slot after slot;
- a sensor that **drifts**: its readings creep up a little more each day although
  the room has not changed, so a fixed alarm level would go off more and more often.

1. `VERIFY` **Run the cell below.** It draws four sensor channels over the sixty days.
2. `VERIFY` **Read the paragraph under the charts** (it starts *What to look for*).
   Then find each of the two faults it describes in the four charts.
3. `PROMPT` `GENERATE` `VERIFY` **Confirm the flat stretch with Gemini**, even if you think you saw it: sixty days
   on one axis hides details. Start a new chat (**⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), then copy the
   quoted text below into the chat box (drag across it from *Rules:* to *starts.*, Ctrl+C, then Ctrl+V in the chat box) and press Enter:

   > **Rules:** Do not change grid. Explain in plain words what the code does.
   >
   > **Task:** For each of temp_c, humidity_pct, co2_ppm and energy_kwh in grid, find the
   > longest run of identical consecutive values, its length in rows, and the date it
   > starts.

   Gemini adds a new code cell with a green edge. Click **Accept and run** at the
   bottom of the chat panel. The answer is printed under that green cell (Colab
   scrolls you there; scroll back to Task 5 afterwards). The stuck sensor is the one
   whose run is about 168 rows (14 hours of 5-minute readings); short runs of a few
   rows are ordinary quiet stretches.
4. **Write two answers** in the box at the end of this task:
   1. Which channel was stuck, on which date, and for how many rows? (Gemini's answer:
      the run of about 168 rows, its channel and the date it starts)
   2. Which channel drifted, and from about which date? (the chart whose line climbs in
      the last weeks and does not come back down)

In [ ]:
# Draw all four channels. The finding is yours; the drawing is not the exercise.
if filled(grid):
    for col in ["temp_c", "humidity_pct", "co2_ppm", "energy_kwh"]:
        show.series(grid, col=col, title=col)

**What to look for.** You should see four charts, one per channel. In one channel, a stretch that is perfectly flat; in another, a slow
climb over the last two and a half weeks that never comes back down. Neither is out of
range, neither is a duplicate, neither is missing. Every check in the library
passes on both.

**Stuck sensor (channel, dates, what it was doing):**

**Drifting sensor (channel, dates, what it was doing):**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Stuck: `temp_c` on 14 May. It reads exactly 23.87 °C from 09:00 to 22:55, which is
168 slots in a row, about 14 hours. A real room warms up through a school day and
cools at night; this line does not move at all. Drifting: `co2_ppm` from about 12
June to the end of the file on 29 June. At night, when the room is empty, it reads
about 420 ppm (outdoor air) until 10 June, then creeps up to about 570 ppm by 28 June
and never comes back. Outdoor air has
not changed, so the sensor has. A common wrong answer names the `energy_kwh` spikes
or the 11 June outage. Those are faults, but the checks and the table already showed
them. This box is for the two you could only see by plotting.

**Both faults on a chart**, slowly (about a minute):
![The stuck temperature sensor on 14 May and the drifting CO2 sensor in June: normal values, so no check fires](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_silent_faults.gif)
</details>

---
## Predict: How many dry days next week? (15 minutes)

*The Prediction League: one question per lab, settled by reality. Individual work: you enter on your own, under your own name.*

**The question:** On how many of the seven days of the target week the island-average rain is zero (under 0.05 mm). The target is the week of Monday 2026-11-16.

**How it works:** you enter a short function, `predict(history)`, not a number. When
entries close, the league runs your function once on the latest data. When the target
is over, the real value is compared with yours. League page (with the closing time):
https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=4

**Step 1.** **Run the cell below.** It prints the number from the simple rule,
*the share of dry days in the last sixty days, times seven, rounded*, on the lab's own file. The league works the simple rule out again
from the latest data when entries close, so its number will differ.

In [ ]:
fetch("sg_rainfall_daily.csv")
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])

def predict(history):
    # the baseline: the share of dry days in the last sixty days, times seven
    dry = history["rain_mm_mean"].tail(60) < 0.05
    print("dry days in the last sixty:", int(dry.sum()))
    return float(round(dry.mean() * 7))

print("The baseline predicts", round(predict(rain), 1), "dry days")

**What you should see:** *The baseline predicts 1.0 dry days*.

**Step 2.** **Ask Gemini to write your predict.** Start a new
chat (**⋮** › **New chat** › **Yes**), paste the prompt and press Enter.

> **Rules:** Put it in a new cell and do not change the cells above. Write only the function and any import lines it needs. Keep the name *predict* and its one input, *history*, a pandas table with the columns date, rain_mm_mean (island average, mm), rain_mm_max (wettest gauge, mm) and stations_reporting, oldest first. Return one number with float(...). **Task:** Write a predict(history) that estimates this: on how many of the seven days of the target week the island-average rain is zero (under 0.05 mm). Do it in a way you think beats the simple rule (the share of dry days in the last sixty days, times seven, rounded). Explain the idea in one sentence.

Click **Accept and run** on Gemini's green cell. It prints nothing; it only defines the
function. If Gemini shows red and green lines inside the Step 1 cell, click **Cancel** and type *put it in a new cell*.

**Step 3.** **Run the cell below.**

In [ ]:
# TODO(you): run Gemini's green cell first (Step 2)
show_predict(rain, "dry days")

**What you should see:** *Your predict says* and a number, usually between 0 and 7, then your
code between two dashed lines. If it says *Gemini's predict has not run yet*, do what that message says.
If you see a red error, paste the last red line into Gemini with *fix this*, click
**Accept and run**, and run this cell again.

<details><summary><b>Using ChatGPT, Claude or another AI instead of Gemini?</b> Open this for a prompt to copy</summary>

That AI cannot see this notebook, so it needs a longer prompt that spells out every
rule the league checks. Use it in place of Steps 2 and 3:

1. **Copy the prompt** in the grey box below.
2. **Paste it** into the other AI's chat box. Do not send it yet.
3. **Write your idea.** In the line *My idea: WRITE YOUR IDEA HERE*, replace *WRITE YOUR IDEA HERE* with your idea, in plain words.
   Plain text is fine: no bold or italics needed. If you have no idea yet, delete the whole line: the AI then chooses one.
4. **Optional:** to let the AI look at the numbers first, download the data,
   https://stanleysp-eng.github.io/EP0418-labs/data/rain_daily.csv (the same file your
   function receives, updated every night), and attach it to the chat.
5. **Send it.** Copy the code the AI gives back, then do **Step 4** below: paste that
   code into the league page's **Code** box, in place of the code from Step 3.

```text
I am entering a prediction contest. Write one Python function for it.

The question: On how many of the seven days of the target week the island-average rain is zero (under 0.05 mm). The target is the week of Monday 2026-11-16 to Sunday 2026-11-22.
The simple rule to beat: the share of dry days in the last sixty days, times seven, rounded.
My idea: WRITE YOUR IDEA HERE

The contest runs the function automatically, so follow these rules exactly:
1. Name it predict, with exactly one input called history:  def predict(history):
2. history is a pandas DataFrame that the contest passes in, oldest row first. Its column names are exactly: date, rain_mm_mean, rain_mm_max, stations_reporting. What they hold: date, rain_mm_mean (island average, mm), rain_mm_max (wettest gauge, mm) and stations_reporting. The date column is a pandas datetime, one row per day, with no time zone.
3. Do not read any file and do not ask for input. All the data is already in history. If I attach a data file, look at it only to choose the idea; the function must still take everything from history.
4. Return one number with return float(...). Not a list, a table, text, None or NaN.
5. Use only pandas, numpy and the Python standard library, and finish in under 30 seconds.
6. Give only the import lines and the function: no example data, no test code, nothing that calls predict.

After the code, explain the idea in one sentence.
```
</details>

**Step 4. Submit on the league page.**
1. **Copy your code** from the Step 3 output: everything between the two dashed lines.
2. **Open** https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=4 in a new tab and type your name as on the class list (the same name every week).
3. **Paste** it into the **Code** box: click in it, Ctrl+A, Ctrl+V (Cmd on a Mac).
4. **Click Run tests** (about 15 seconds the first time). **What you should see:** *All 4 cases ran* and four tabs: three past cases and *Your entry*. On *Your entry*, Actual says *not known yet*, and the number differs from Colab's because the page uses newer data: both are normal. A red *missed by* only means the simple rule did better on that case. A **red heading** instead of *All 4 cases ran* is an error: copy the red text under the case marked red (or under the heading), paste it into the same Gemini chat with *fix this*, click **Accept and run**, run Step 3 again, then paste and **Run tests** again.
5. **Click Submit.** *Submitted at ...* confirms it. Submit again any time before the close; the latest one counts.

**Write it down** (on the sheet, or double-click here): your function's number (Step 3),
the baseline (Step 1), and the one-sentence idea Gemini gave. After the target, the
real answer is on the league page.

**My number / the baseline / the idea:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

There is no single right answer here. A good entry gives all three parts: your
function's number, the baseline (1.0 dry days), and an idea that uses the data. 11 of the 60 days in the file were dry, so the baseline says 1 (1.28, rounded). Dry days come in runs, so a week can have none or several. Ideas that use this: count the dry days in each recent week and average those, or weight the last two weeks more.
The common weak answer is "Gemini wrote it". It says who wrote the code, not what the idea is. The past-case
tabs on the league page show whether the idea would have beaten the simple rule
lately; three cases are too few to prove it.
</details>

---
## Read before you run

An assistant was asked to put the cleaned sensor data onto a 5-minute grid with no blank values left. It wrote the code below.

The code is switched off: every line starts with `#`, so it cannot run yet.

1. **Read it. Do not run it.** Answer questions 1 and 2 in the box under the code
   cell, or on the sheet.
2. **Ask Gemini to explain it.** Click in the code cell, press Ctrl+A then Ctrl+C
   (Cmd on a Mac). In Gemini, start a new chat (**⋮** › **New chat** › **Yes**),
   paste the code into the chat box, add *Explain this code in plain words.* and
   press Enter. If Gemini offers a green cell or red and green lines, click
   **Cancel**: this step only explains. Check whether the explanation points out the problem.
   Often it does not.
3. **Then run it.** Delete the `# ` at the start of each line below the first line,
   run the cell, and see whether you were right.

**Question 1.** What does the `interpolate()` line do to the five-and-a-half-hour outage on 11 June?

**Question 2.** No arithmetic needed: predict whether the second blank count will be zero, smaller than the first, or the same, and whether the Task 3 checks pass. At step 3 you run it: it prints both counts and runs those two checks on its result. (It uses its own name, `their_grid`, so your `grid` is untouched.)

<details><summary>Hint 1: where to look</summary>

What does `interpolate()` do to a run of empty slots between two real readings?

</details>
<details><summary>Hint 2: closer to the answer</summary>

It draws a straight line through them. The 66 empty slots of the 11 June outage get invented values, and so does every blank humidity value. The row count does not change, so `rows_accounted_for` cannot see it, and the Task 3 checks still pass.

![interpolate() draws a straight line through the 11 June outage: 66 invented readings, and every check still passes](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_interpolate.gif)

</details>

An explanation that reads well is not evidence the code is right. 

**Optional, to fix it:** in the same chat, type *Fix the problem in this code that
Hint 2 describes. Say in one sentence what you changed.* If that one change matches
Hint 2, click **Accept and run** (accepting is right here: you asked for the fix). If
it changed other things too, click **Cancel** and type *fix only that, nothing else*.
You should see: the second blank count is no longer 0: the outage and the humidity blanks stay blank.

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# their_grid = clean.set_index("timestamp").resample("5min").mean()
# print("blank values before interpolate:", int(their_grid.isna().sum().sum()))
# their_grid = their_grid.interpolate()
# print("blank values after interpolate :", int(their_grid.isna().sum().sum()))
# check.reset(); check.regular_cadence(their_grid, "5min"); check.values_in_range(their_grid.reset_index(), "temp_c", 15, 40); check.report()

**Answer to question 1:**

**Answer to question 2:**

**Did Gemini's explanation mention the fault? (yes / no, and the sentence):**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks tested the code. This quiz tests what you understood. In Gemini, start a new chat
(**⋮** › **New chat** › **Yes**), paste this into the chat box and press Enter:

> **Rules:** Ask one short question at a time, in plain words. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer.
>
> **Task:** Quiz me on this lab: five questions on these ideas: the ways a missing value was disguised in today's file (-999, blank, missing rows); why duplicate timestamps appear and which copy to keep; why every row you remove must be accounted for; what rows_accounted_for checks and what it cannot see; what a stuck sensor looks like and why no check catches it. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks one question at a time in the chat panel; answer in the chat box, in
your own words, without looking back. Ignore the suggested replies Gemini shows. If you
use ChatGPT, Claude or Copilot instead, first delete the sentence *Take the examples from
the code and outputs in this notebook.*: they cannot see your notebook.

Then write your score and the idea to revise below. Your tutor asks for them.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

*(Your answer box: double-click this cell. Type your answer here.)*

---
## Before you leave

Double-click the box cell below and type there (or write on the last page of the Lab 4
sheet): one **mistake caught** line per AI mistake you caught today, and one
**decision** line for each fault you handled (the −999s, the duplicates, the outage
gap), saying what you did and why. The Task 2 expected output gives one example.

**Mistake caught:** *(three short lines: 1. what I expected, 2. what was there instead, 3. how I found it. If Gemini made no mistake today, use the read-before-you-run code above instead.)*

**Decision:** *(what I decided · why · the evidence)*

*(Your answer box: double-click this cell. Type your answer here.)*